In [ ]:
note_nr = 96

filename = "note_reference_" + str(note_nr)
print(filename)

piano = "Shigeru_Kawai_SKEX"

def set_filenames():
    global filename
    global piano  
    global originalMIDI
    global originalTXT
    global originalPNG
    global originalAUDIO
    global generatedMIDI
    global generatedTXT
    global generatedPNG
    global generatedAUDIO
    originalMIDI = filename + ".midi"
    originalTXT = filename + ".txt"
    originalPNG = filename + ".png"
    originalAUDIO = filename + ".wav"
    generatedMIDI = filename + ""
    generatedTXT = filename + "_out.txt"
    generatedPNG = filename + ""
    generatedAUDIO = filename + "_out.wav"
set_filenames()
print("original MIDI filename:", originalMIDI)
print("original TXT filename:", originalTXT)
print("original PNG filename:", originalPNG)
print("original AUDIO filename:", originalAUDIO)
print("generated MIDI filename:", generatedMIDI)
print("generated TXT filename:", generatedTXT)
print("generated PNG filename:", generatedPNG)
from IPython.display import Audio


In [ ]:
import os
import subprocess
import random
from mido import Message, MidiFile, MidiTrack, MetaMessage

def generate_midi(note_nr, random_flag=0):
    """
    Generates a MIDI file (and renders a WAV file using fluidsynth) for a given MIDI note.
    
    Parameters:
      note_nr (int): MIDI note number.
      random_flag (int): If set to 1, velocities are shuffled.
    """
    # Hardcoded timing values (ticks)
    note_duration = 288  # on duration: 300ms
    gap_duration = 192   # off duration: 200ms


    # Create a filename based on the parameters.
    if random_flag == 1:
        filename = "note_reference_random_" + str(note_nr)
    else:
        filename = "note_reference_" + str(note_nr)
    
   
    # Define filenames.
    originalMIDI = filename + ".midi"
    originalAUDIO = filename + ".wav"
    
    # Create a new MIDI file and add two tracks.
    mid = MidiFile(ticks_per_beat=480)
    piano_track = MidiTrack()
    pedal_track = MidiTrack()
    mid.tracks.append(piano_track)
    mid.tracks.append(pedal_track)
    
    # Set a tempo (500000 microseconds per beat corresponds to 120 BPM).
    piano_track.append(MetaMessage('set_tempo', tempo=500000, time=0))
    
    # Generate a list of MIDI velocity values (0 to 127) and shuffle if random_flag is enabled.
    velocities = list(range(128))
    if random_flag == 1:
        random.shuffle(velocities)
    
    # Insert an initial note with velocity 127.
    piano_track.append(Message('note_on', note=note_nr, velocity=127, time=0))
    piano_track.append(Message('note_off', note=note_nr, velocity=0, time=note_duration))
    
    # For each velocity, add the note with the gap and note duration.
    for i, velocity in enumerate(velocities):
        piano_track.append(Message('note_on', note=note_nr, velocity=velocity, time=gap_duration))
        piano_track.append(Message('note_off', note=note_nr, velocity=0, time=note_duration))
        if i < len(velocities) - 1:
            piano_track.append(Message('note_on', note=note_nr, velocity=127, time=gap_duration))
            piano_track.append(Message('note_off', note=note_nr, velocity=0, time=note_duration))
    
    # Save the MIDI file.
    mid.save(originalMIDI)
    print(f"MIDI file '{originalMIDI}' generated successfully!")
    
    # Render the MIDI file to a WAV file using fluidsynth with the specified soundfont.
    # Update the soundfont path as needed.
    fluidsynth_cmd = f"fluidsynth -ni /Users/user/.soundfonts/Shigeru_Kawai_SKEX.sf2 {originalMIDI} -F {originalAUDIO} -r 44100 -g 1"
    subprocess.run(fluidsynth_cmd, shell=True)
    print(f"WAV file '{originalAUDIO}' generated successfully!")

generate_midi(note_nr=note_nr, random_flag=0)


In [ ]:
def midi_to_text_and_plot(midi_path, text_path, plot_path, title_text):
    """
    Decodes a MIDI file, writes its events into a text file, and plots:
    1) Velocity vs. Time (ignoring velocity 0) with Y-axis from 0 to 127
    2) Histogram of Velocity Values with 128 bins and X-axis from 0 to 127
    3) MIDI Note Numbers vs. Time mapped to piano keys (1-88)
    4) Histogram of MIDI Note Numbers (mapped to 1-88) with 88 bins
    Args:
        midi_path (str): Path to the input MIDI file.
        text_path (str): Path to save the output text file.
        plot_path (str): Path to save the generated plot as a PNG file.
        title_text (str): Text to be displayed in the upper left corner of the plot.
    """
    import mido
    import matplotlib.pyplot as plt

    midi_file = mido.MidiFile(midi_path)
    ticks_per_beat = midi_file.ticks_per_beat
    tempo = 500000  # Default 120 BPM tempo (500,000 microseconds per beat)

    # Store extracted data
    velocity_values = []
    velocity_times = []
    note_numbers = []
    note_times = []

    with open(text_path, 'w') as text_file:
        text_file.write(f"MIDI File: {midi_path}\n")
        text_file.write(f"Ticks per Beat: {ticks_per_beat}\n\n")

        absolute_ticks = 0  # Tracks cumulative time in ticks

        for i, track in enumerate(midi_file.tracks):
            text_file.write(f"Track {i}: {track.name}\n")
            text_file.write("-" * 40 + "\n")

            for msg in track:
                absolute_ticks += msg.time  # Convert relative to absolute time

                # Tempo Change Handling (Optional)
                if msg.type == 'set_tempo':
                    tempo = msg.tempo  # Set new tempo

                # Convert MIDI ticks to seconds
                seconds = mido.tick2second(absolute_ticks, ticks_per_beat, tempo)

                # Note-On Event (Velocity > 0)
                if msg.type == 'note_on' and msg.velocity > 0:
                    velocity_values.append(msg.velocity)  # Store velocity
                    velocity_times.append(seconds)         # Store time for velocity
                    note_numbers.append(msg.note)            # Store MIDI note number
                    note_times.append(seconds)               # Store time for note

                    text_file.write(f"Time {seconds:.3f} sec | NOTE_ON  | "
                                    f"Note: {msg.note:3d} | Velocity: {msg.velocity:3d}\n")

                # Note-Off Event
                elif msg.type == 'note_off' or (msg.type == 'note_on' and msg.velocity == 0):
                    text_file.write(f"Time {seconds:.3f} sec | NOTE_OFF | "
                                    f"Note: {msg.note:3d} | Velocity: 0\n")

            text_file.write("\n")

    # Map MIDI note numbers to piano keys: MIDI 21 -> 1, MIDI 108 -> 88.
    mapped_note_numbers = [n - 20 for n in note_numbers]

    # Generate Plots as Subplots if there is any velocity data
    if velocity_values:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        # Top Left: Velocity vs. Time (set Y-axis from 0 to 127)
        axes[0, 0].scatter(velocity_times, velocity_values, color='b', alpha=0.7, label="Note Velocity")
        axes[0, 0].set_xlabel("Time (seconds)")
        axes[0, 0].set_ylabel("Velocity (0-127)")
        axes[0, 0].set_title("Velocity vs. Time")
        axes[0, 0].legend()
        axes[0, 0].grid()
        axes[0, 0].set_ylim(0, 127)
        axes[0, 0].set_xlim(left=0)
        axes[0, 0].text(-0.07, 1.1, title_text, transform=axes[0, 0].transAxes, fontsize=12, verticalalignment='top')

        # Bottom Left: MIDI Note Numbers vs. Time (mapped to 1-88)
        axes[1, 0].scatter(note_times, mapped_note_numbers, color='purple', alpha=0.7, label="Piano Key")
        axes[1, 0].set_xlabel("Time (seconds)")
        axes[1, 0].set_ylabel("Piano Key (1-88)")
        axes[1, 0].set_title("MIDI Note Numbers vs. Time")
        axes[1, 0].legend()
        axes[1, 0].grid()
        axes[1, 0].set_xlim(left=0)
        axes[1, 0].set_ylim(1, 88)

        # Top Right: Histogram of Velocity Values (set X-axis from 0 to 127)
        axes[0, 1].hist(velocity_values, bins=128, range=(0, 127), color='g', alpha=0.7, edgecolor='black')
        axes[0, 1].set_xlabel("Velocity")
        axes[0, 1].set_ylabel("Number of Occurrences")
        axes[0, 1].set_title("Histogram of Velocity Values")
        axes[0, 1].grid()
        axes[0, 1].set_xlim(0, 127)

        # Bottom Right: Histogram of MIDI Note Numbers (mapped to 1-88)
        axes[1, 1].hist(mapped_note_numbers, bins=88, range=(1, 88), color='orange', alpha=0.7, edgecolor='black')
        axes[1, 1].set_xlabel("Piano Key (1-88)")
        axes[1, 1].set_ylabel("Number of Occurrences")
        axes[1, 1].set_title("Histogram of MIDI Note Numbers")
        axes[1, 1].grid()
        axes[1, 1].set_xlim(1, 88)

        plt.tight_layout()
        plt.savefig(plot_path)  # Save plot as PNG file
        plt.show()

    print(f"PLOTS saved to: {plot_path}")
    print(f"MIDI events saved to: {text_path}")
midi_to_text_and_plot(originalMIDI, originalTXT, originalPNG, filename )

In [ ]:
print(originalAUDIO)
Audio(originalAUDIO)

In [ ]:
generatedMIDI = filename + "_" + piano + ".mid"
generatedPNG = filename + "_" + piano + ".png"
generatedTXT = filename + "_" + piano + ".txt"
print(generatedMIDI)
print(generatedPNG)
print(generatedTXT)
!python example.py --audio_path={originalAUDIO} --output_midi_path={generatedMIDI}

In [ ]:
#Filtering out all MIDI events which are not related to the original, reference MIDI notes

try:
    with open("output.txt", "r") as infile:
        lines = infile.readlines()
except FileNotFoundError:
    print("The file 'output.txt' does not exist in the current directory.")
else:
    # Filter lines where the second column equals note_nr
    filtered_lines = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) >= 2:
            try:
                # Convert the second column value to an integer
                second_col_value = int(parts[1])
            except ValueError:
                # Skip the line if conversion fails
                continue
            if second_col_value == note_nr:
                filtered_lines.append(line.strip())

    # Write the filtered lines to the file represented by generatedTXT
    with open(generatedTXT, "w") as outfile:
        for line in filtered_lines:
            outfile.write(line + "\n")

    print(f"{len(filtered_lines)} lines have been written to '{generatedTXT}'.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_velocity_mapping(filename=generatedTXT, degree=3):
    # Load the data
    data = np.loadtxt(filename)

    # Extract columns
    original_vals = data[:, 0]
    inferred_vals = data[:, 2]
    inferred_midi = data[:, 3]

    # Fit polynomial
    coeffs = np.polyfit(inferred_vals, original_vals, deg=degree)
    fitted_originals = np.polyval(coeffs, inferred_vals)

    # Calculate R^2
    residuals = original_vals - fitted_originals
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((original_vals - np.mean(original_vals))**2)
    r2_score = 1 - (ss_res / ss_tot)

    # Sort data
    sort_idx = np.argsort(inferred_vals)
    sorted_inferred = inferred_vals[sort_idx]
    sorted_original = original_vals[sort_idx]
    sorted_fitted = np.polyval(coeffs, sorted_inferred)

    sort_idx2 = np.argsort(original_vals)
    sorted_original2 = original_vals[sort_idx2]
    sorted_fitted2 = fitted_originals[sort_idx2]
    sorted_inferred_midi = inferred_midi[sort_idx2]

    fit_error = sorted_fitted2 - sorted_original2
    inferred_error = sorted_inferred_midi - sorted_original2

    # Plot
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

    ax1.scatter(sorted_original, sorted_inferred, color='blue', alpha=0.7, label='Original vs Inferred')
    ax1.plot(sorted_fitted, sorted_inferred, color='red', linewidth=2, label='Best-Fit')
    ax1.set_ylabel("inferred Velocity (0:1)")
    ax1.set_xlabel("original MIDI Velocity")
    ax1.set_title(f"Best-Fit Original → Inferred\n(Poly Degree {degree}, R²={r2_score:.3f})")
    ax1.text(20.0, 0.97, generatedTXT, fontsize=12, verticalalignment='top')
    ax1.grid(True)
    ax1.legend()

    ax2.plot(sorted_original2, sorted_fitted2, marker='o', linestyle='-', color='green', label='Best-Fit MIDI Values')
    ax2.plot(sorted_original2, sorted_inferred_midi, marker='s', linestyle='-', color='orange', label='Inferred MIDI Values')
    min_val, max_val = np.min(original_vals), np.max(original_vals)
    ax2.plot([min_val, max_val], [min_val, max_val], color='black', linestyle='--', label='y = x')
    ax2.set_xlabel("original MIDI Velocity")
    ax2.set_ylabel("infered MIDI Velocity (0:127)")
    ax2.set_title("Best-Fit & Inferred MIDI Velocity vs. Original MIDI Velocity")
    ax2.grid(True)
    ax2.legend()

    ax3.plot(sorted_original2, fit_error, marker='o', linestyle='-', color='green', label='Best-Fit Error')
    ax3.plot(sorted_original2, inferred_error, marker='o', linestyle='-', color='orange', label='Inferred Velocity Error')
    ax3.axhline(0, color='black', linestyle='--', label='Zero Error')
    ax3.set_xlabel("original MIDI Velocity")
    ax3.set_ylabel("Error")
    ax3.set_title("Best-Fit & Inferred MIDI Velocity Error vs. Original MIDI Velocity")
    ax3.grid(True)
    ax3.legend()
    
    plt.savefig(generatedPNG)  # Save plot as PNG file
    plt.tight_layout()
    plt.show()
    print(f"PLOTS saved to: {generatedPNG}")

plot_velocity_mapping()
